In [1]:
!pip install -q gradio matplotlib numpy

import gradio as gr
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon, Rectangle
import numpy as np

# Motor presets: (name, avg_thrust_N, burn_time_s, propellant_mass_kg)
MOTORS = {
    "A10 (1.88 Ns)": {"thrust": 3.2, "burn_time": 0.60, "prop_mass": 0.0025},
    "B6 (5.0 Ns)":  {"thrust": 6.5, "burn_time": 0.80, "prop_mass": 0.0055},
    "C6 (10.0 Ns)": {"thrust": 10.5, "burn_time": 1.00, "prop_mass": 0.0100},
}


def nose_profile_points(nose_type: str, x_start: float, length_cm: float, radius_cm: float, n_pts: int = 40):
    """Generates (x, y_upper) profile coordinates for the selected nose cone geometry."""
    t = np.linspace(0.0, 1.0, n_pts)
    x = x_start + t * length_cm

    if nose_type == "Conical":
        y = radius_cm * t
    elif nose_type == "Parabolic":
        y = radius_cm * np.sqrt(t)
    elif nose_type == "Elliptical":
        y = radius_cm * np.sqrt(np.clip(1.0 - (1.0 - t) ** 2, 0.0, 1.0))
    else:  # Tangent Ogive
        rho = (radius_cm ** 2 + length_cm ** 2) / (2.0 * radius_cm)
        y = np.sqrt(np.clip(rho ** 2 - (length_cm * (1.0 - t)) ** 2, 0.0, None)) - (rho - radius_cm)
        y[0] = 0.0
        y[-1] = radius_cm

    return x, y


def draw_rocket_2d(
    nose_type, nose_len_cm, body_len_cm, body_diam_cm,
    fin_type, fin_root_cm, fin_tip_cm, fin_span_cm, fin_sweep_cm,
    cg_cm, cp_cm, dry_mass_g, motor_name, stability_cal, apogee_m
):
    fig, ax = plt.subplots(figsize=(11, 3.8), dpi=130)
    fig.patch.set_facecolor("#0f172a")
    ax.set_facecolor("#0f172a")

    r_cm = body_diam_cm / 2.0
    total_len_cm = nose_len_cm + body_len_cm

    # 1. Nose Cone
    nx, ny = nose_profile_points(nose_type, 0.0, nose_len_cm, r_cm)
    nose_poly = np.vstack([
        np.column_stack([nx, ny]),
        np.column_stack([nx[::-1], -ny[::-1]])
    ])
    ax.add_patch(Polygon(nose_poly, closed=True, facecolor="#1e293b", edgecolor="#38bdf8", lw=1.8))

    # 2. Body Tube
    ax.add_patch(
        Rectangle((nose_len_cm, -r_cm), body_len_cm, 2.0 * r_cm,
                  facecolor="#1e3a5f", edgecolor="#93c5fd", lw=1.6)
    )

    # Internal structure: Recovery bay & Motor mount
    chute_x = nose_len_cm + 0.15 * body_len_cm
    chute_w = 0.22 * body_len_cm
    ax.add_patch(Rectangle((chute_x, -0.65 * r_cm), chute_w, 1.3 * r_cm,
                           fill=False, linestyle="--", edgecolor="#facc15", lw=1.0))
    ax.text(chute_x + chute_w / 2, 0, "Recovery", color="#fde047", fontsize=7, ha="center", va="center")

    motor_w = min(0.24 * body_len_cm, 9.0)
    motor_x = total_len_cm - motor_w - 0.5
    ax.add_patch(Rectangle((motor_x, -0.55 * r_cm), motor_w, 1.1 * r_cm,
                           facecolor="#334155", edgecolor="#cbd5e1", lw=1.1))
    ax.text(motor_x + motor_w / 2, 0, motor_name.split()[0], color="#e2e8f0", fontsize=7, ha="center", va="center")

    # 3. Fins (Top and Bottom 2D cross-section)
    fin_start_x = total_len_cm - fin_root_cm - 0.5
    for sign in [1.0, -1.0]:
        if fin_type == "Elliptical":
            ft = np.linspace(0.0, np.pi, 24)
            fx = fin_start_x + (fin_root_cm / 2.0) * (1.0 - np.cos(ft)) + fin_sweep_cm * np.sin(ft) * 0.5
            fy = sign * (r_cm + fin_span_cm * np.sin(ft))
            pts = np.column_stack([fx, fy])
        else:  # Trapezoidal
            pts = np.array([
                [fin_start_x, sign * r_cm],
                [fin_start_x + fin_sweep_cm, sign * (r_cm + fin_span_cm)],
                [fin_start_x + fin_sweep_cm + fin_tip_cm, sign * (r_cm + fin_span_cm)],
                [fin_start_x + fin_root_cm, sign * r_cm],
            ])
        ax.add_patch(Polygon(pts, closed=True, facecolor="#2563eb", edgecolor="#bfdbfe", lw=1.5))

    # Centerline
    ax.axhline(0.0, color="#64748b", linestyle=":", lw=1.0)

    # 4. CG (Blue) and CP (Red) markers
    y_arrow = r_cm + fin_span_cm * 0.55 + 1.2
    ax.scatter([cg_cm], [0.0], color="#38bdf8", s=95, zorder=6, edgecolors="white", lw=1.2)
    ax.annotate(f"CG: {cg_cm:.1f} cm", xy=(cg_cm, 0.0), xytext=(cg_cm, y_arrow),
                color="#7dd3fc", fontsize=8.5, fontweight="bold", ha="center",
                arrowprops=dict(arrowstyle="->", color="#7dd3fc", lw=1.0))

    ax.scatter([cp_cm], [0.0], color="#f43f5e", s=95, zorder=6, edgecolors="white", lw=1.2)
    ax.annotate(f"CP: {cp_cm:.1f} cm", xy=(cp_cm, 0.0), xytext=(cp_cm, -(y_arrow + 0.6)),
                color="#fda4af", fontsize=8.5, fontweight="bold", ha="center",
                arrowprops=dict(arrowstyle="->", color="#fda4af", lw=1.0))

    # Summary box (Top right)
    info_txt = (
        f"Length: {total_len_cm:.1f} cm | Diameter: {body_diam_cm:.1f} cm\n"
        f"Dry Mass: {dry_mass_g:.0f} g | Stability: {stability_cal:.2f} cal\n"
        f"Est. Apogee: ~{apogee_m:.1f} m"
    )
    ax.text(0.99, 0.96, info_txt, transform=ax.transAxes, fontsize=8, color="#f8fafc",
            ha="right", va="top", bbox=dict(boxstyle="round,pad=0.4", facecolor="#1e293b", edgecolor="#475569"))

    y_lim = max(r_cm + fin_span_cm + 3.0, 7.0)
    ax.set_xlim(-2.0, total_len_cm + 3.0)
    ax.set_ylim(-y_lim, y_lim)
    ax.set_xlabel("Axial Position from Nose Tip (cm)", color="#cbd5e1", fontsize=9)
    ax.set_ylabel("Radial Offset (cm)", color="#cbd5e1", fontsize=9)
    ax.set_title("2D Rocket Layout, Internal Structure, and CG / CP Locations", color="#f1f5f9", fontsize=10.5, pad=8)
    ax.tick_params(colors="#94a3b8", labelsize=8)
    for spine in ax.spines.values():
        spine.set_color("#334155")

    plt.tight_layout()
    return fig


def simulate_stability_over_time(
    nose_type, nose_len_cm, body_len_cm, body_diam_cm,
    fin_type, fin_root_cm, fin_tip_cm, fin_span_cm, fin_sweep_cm,
    cg_cm, cp_cm, dry_mass_g, motor_name, wind_ms
):
    m_info = MOTORS[motor_name]
    burn_t = m_info["burn_time"]
    thrust = m_info["thrust"]
    prop_mass_kg = m_info["prop_mass"]
    mass_kg = max(dry_mass_g / 1000.0, 0.03)

    # Static stability margin in calibers (body diameters)
    base_stab = (cp_cm - cg_cm) / max(body_diam_cm, 0.5)

    # Fin area, span, and wind effects
    fin_area_factor = ((fin_root_cm + fin_tip_cm) * 0.5 * fin_span_cm) / 120.0
    aspect_penalty = max(0.0, (fin_span_cm - 9.0) * 0.025)
    wind_effect = 0.03 * wind_ms

    # Time vector covering burn, coast, apogee, and early descent
    t_flight = burn_t + 5.5
    t = np.linspace(0.0, t_flight, 220)

    burn_progress = np.clip(t / burn_t, 0.0, 1.0)
    cg_shift_cm = 0.4 * (prop_mass_kg * 1000.0) * burn_progress * 0.25

    apogee_t = burn_t + 2.2
    peak_dip = -0.22 * np.exp(-((t - apogee_t) / 0.7) ** 2)
    dynamic_mod = (
        0.12 * fin_area_factor
        - aspect_penalty
        - wind_effect * (0.4 + 0.6 * (t / t_flight))
        + peak_dip
        - (cg_shift_cm / max(body_diam_cm, 0.5)) * 0.35
    )

    stab_t = base_stab + dynamic_mod

    # Approximate apogee estimate
    impulse = thrust * burn_t
    v_est = impulse / (mass_kg + 0.5 * prop_mass_kg)
    drag_factor = 1.0 + 0.04 * (body_diam_cm - 3.5) + 0.015 * fin_span_cm
    apogee_m = max(5.0, (v_est ** 2) / (2.0 * 9.81 * drag_factor))

    fig_2d = draw_rocket_2d(
        nose_type, nose_len_cm, body_len_cm, body_diam_cm,
        fin_type, fin_root_cm, fin_tip_cm, fin_span_cm, fin_sweep_cm,
        cg_cm, cp_cm, dry_mass_g, motor_name, base_stab, apogee_m
    )

    fig_t, ax_t = plt.subplots(figsize=(11, 3.6), dpi=130)
    fig_t.patch.set_facecolor("#0f172a")
    ax_t.set_facecolor("#0f172a")

    ax_t.axhspan(1.0, 3.0, color="#10b981", alpha=0.14, label="Target Stability Band (1.0–3.0 cal)")
    ax_t.axhline(1.0, color="#f59e0b", linestyle="--", lw=1.1)
    ax_t.axhline(3.0, color="#38bdf8", linestyle="--", lw=1.1)
    ax_t.axhline(0.0, color="#ef4444", linestyle=":", lw=1.1)

    ax_t.plot(t, stab_t, color="#a78bfa", lw=2.2, label="Time-Varying Stability (cal)")
    ax_t.axvline(burn_t, color="#fb923c", linestyle="-.", lw=1.1, label=f"Motor Burnout ({burn_t:.2f} s)")
    ax_t.axvline(apogee_t, color="#34d399", linestyle="-.", lw=1.1, label=f"Est. Apogee (~{apogee_t:.1f} s)")

    ax_t.set_xlabel("Flight Time, t (s)", color="#cbd5e1", fontsize=9)
    ax_t.set_ylabel("Stability Margin (cal)", color="#cbd5e1", fontsize=9)
    ax_t.set_title("Stability Margin Over Time", color="#f1f5f9", fontsize=10.5, pad=8)
    ax_t.tick_params(colors="#94a3b8", labelsize=8)
    ax_t.legend(loc="upper right", fontsize=7.5, facecolor="#1e293b", edgecolor="#475569", labelcolor="#f8fafc", ncol=2)
    for spine in ax_t.spines.values():
        spine.set_color("#334155")

    plt.tight_layout()

    min_stab = float(np.min(stab_t))
    max_stab = float(np.max(stab_t))

    if min_stab < 0.8 or base_stab < 0.8:
        status = "🔴 UNSTABLE"
        comment = (
            f"**Status — Unstable:** Static stability margin is **{base_stab:.2f} cal** "
            f"(minimum during flight: **{min_stab:.2f} cal**). CP is not sufficiently aft of CG, "
            f"creating high pitch/yaw instability risk. "
            f"**Recommendation:** Move CG forward (add nose ballast) or increase fin area / shift fins aft."
        )
    elif base_stab > 3.2 or max_stab > 3.5:
        status = "🟡 BORDERLINE (Over-Stable / Wind-Sensitive)"
        comment = (
            f"**Status — Borderline (Over-Stable):** Static stability margin is **{base_stab:.2f} cal** "
            f"(peak: **{max_stab:.2f} cal**). High stability increases wind weathercocking under "
            f"**{wind_ms:.1f} m/s** wind, which can reduce altitude. "
            f"**Recommendation:** Reduce fin span slightly or shift CG slightly aft toward the 1.5–2.5 cal range."
        )
    else:
        status = "🟢 GOOD (Nominal Stability)"
        comment = (
            f"**Status — Good:** Static stability margin is **{base_stab:.2f} cal**, staying within "
            f"**{min_stab:.2f} – {max_stab:.2f} cal** across the flight profile. "
            f"The **{nose_type}** nose cone, **{fin_type}** fins, and **{motor_name}** motor provide balanced stability."
        )

    summary_md = f"### {status}\n\n{comment}"
    return fig_2d, fig_t, summary_md


with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("## 🚀 2D Rocket Design, CG/CP Visualization & Time-Dependent Stability Tool")

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### Component & Geometry Inputs")
            nose_type = gr.Dropdown(["Ogive", "Conical", "Parabolic", "Elliptical"], value="Ogive", label="Nose Cone Shape")
            nose_len_cm = gr.Slider(5.0, 35.0, value=16.0, step=0.5, label="Nose Cone Length (cm)")
            body_len_cm = gr.Slider(15.0, 80.0, value=45.0, step=1.0, label="Body Tube Length (cm)")
            body_diam_cm = gr.Slider(2.0, 12.0, value=4.2, step=0.1, label="Body Diameter (cm)")

            fin_type = gr.Dropdown(["Trapezoidal", "Elliptical"], value="Trapezoidal", label="Fin Shape")
            fin_root_cm = gr.Slider(3.0, 20.0, value=9.0, step=0.5, label="Fin Root Chord (cm)")
            fin_tip_cm = gr.Slider(1.0, 15.0, value=4.0, step=0.5, label="Fin Tip Chord (cm)")
            fin_span_cm = gr.Slider(2.0, 18.0, value=7.5, step=0.5, label="Fin Span (cm)")
            fin_sweep_cm = gr.Slider(0.0, 15.0, value=3.0, step=0.5, label="Fin Sweep Length (cm)")

            gr.Markdown("### Mass, CG/CP & Flight Conditions")
            cg_cm = gr.Slider(5.0, 90.0, value=31.0, step=0.5, label="Center of Gravity — CG from Nose (cm)")
            cp_cm = gr.Slider(5.0, 100.0, value=41.5, step=0.5, label="Center of Pressure — CP from Nose (cm)")
            dry_mass_g = gr.Slider(40.0, 600.0, value=145.0, step=5.0, label="Dry Mass (g)")
            motor_name = gr.Dropdown(list(MOTORS.keys()), value="B6 (5.0 Ns)", label="Motor Selection")
            wind_ms = gr.Slider(0.0, 15.0, value=3.5, step=0.5, label="Wind Speed (m/s)")

            run_btn = gr.Button("Run Simulation & Plot", variant="primary")

        with gr.Column(scale=2):
            plot_2d = gr.Plot(label="2D Rocket Cross-Section & CG/CP")
            plot_time = gr.Plot(label="Stability Over Time")
            analysis_out = gr.Markdown()

    inputs = [
        nose_type, nose_len_cm, body_len_cm, body_diam_cm,
        fin_type, fin_root_cm, fin_tip_cm, fin_span_cm, fin_sweep_cm,
        cg_cm, cp_cm, dry_mass_g, motor_name, wind_ms
    ]

    run_btn.click(fn=simulate_stability_over_time, inputs=inputs, outputs=[plot_2d, plot_time, analysis_out])
    demo.load(fn=simulate_stability_over_time, inputs=inputs, outputs=[plot_2d, plot_time, analysis_out])

demo.launch(share=True)

/tmp/ipykernel_5121/220051742.py:234: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://51d761eec7a3638b13.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [4]:
!pip install -q gradio matplotlib numpy

import gradio as gr
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon, Rectangle
import numpy as np

MOTORS = {
    "A10 (1.88 Ns)": {"thrust": 3.2, "burn_time": 0.60, "prop_mass": 0.0025},
    "B6 (5.0 Ns)":  {"thrust": 6.5, "burn_time": 0.80, "prop_mass": 0.0055},
    "C6 (10.0 Ns)": {"thrust": 10.5, "burn_time": 1.00, "prop_mass": 0.0100},
}


def nose_profile_points(nose_type: str, x_start: float, length_cm: float, radius_cm: float, n_pts: int = 40):
    t = np.linspace(0.0, 1.0, n_pts)
    x = x_start + t * length_cm

    if nose_type == "Conical":
        y = radius_cm * t
    elif nose_type == "Parabolic":
        y = radius_cm * np.sqrt(t)
    elif nose_type == "Elliptical":
        y = radius_cm * np.sqrt(np.clip(1.0 - (1.0 - t) ** 2, 0.0, 1.0))
    else:  # Tangent Ogive
        rho = (radius_cm ** 2 + length_cm ** 2) / (2.0 * radius_cm)
        y = np.sqrt(np.clip(rho ** 2 - (length_cm * (1.0 - t)) ** 2, 0.0, None)) - (rho - radius_cm)
        y[0] = 0.0
        y[-1] = radius_cm

    return x, y


def draw_rocket_2d(
    nose_type, nose_len_cm, body_len_cm, body_diam_cm,
    fin_type, fin_root_cm, fin_tip_cm, fin_span_cm, fin_sweep_cm,
    cg_cm, cp_cm, dry_mass_g, motor_name, stability_cal, apogee_m
):
    fig, ax = plt.subplots(figsize=(10.5, 3.2), dpi=120)
    fig.patch.set_facecolor("#0f172a")
    ax.set_facecolor("#0f172a")

    r_cm = body_diam_cm / 2.0
    total_len_cm = nose_len_cm + body_len_cm

    # 1. Nose Cone
    nx, ny = nose_profile_points(nose_type, 0.0, nose_len_cm, r_cm)
    nose_poly = np.vstack([
        np.column_stack([nx, ny]),
        np.column_stack([nx[::-1], -ny[::-1]])
    ])
    ax.add_patch(Polygon(nose_poly, closed=True, facecolor="#1e293b", edgecolor="#38bdf8", lw=1.8))

    # 2. Body Tube
    ax.add_patch(
        Rectangle((nose_len_cm, -r_cm), body_len_cm, 2.0 * r_cm,
                  facecolor="#1e3a5f", edgecolor="#93c5fd", lw=1.6)
    )

    # Recovery bay & Motor mount
    chute_x = nose_len_cm + 0.15 * body_len_cm
    chute_w = 0.22 * body_len_cm
    ax.add_patch(Rectangle((chute_x, -0.65 * r_cm), chute_w, 1.3 * r_cm,
                           fill=False, linestyle="--", edgecolor="#facc15", lw=1.0))
    ax.text(chute_x + chute_w / 2, 0, "Recovery", color="#fde047", fontsize=7, ha="center", va="center")

    motor_w = min(0.24 * body_len_cm, 9.0)
    motor_x = total_len_cm - motor_w - 0.5
    ax.add_patch(Rectangle((motor_x, -0.55 * r_cm), motor_w, 1.1 * r_cm,
                           facecolor="#334155", edgecolor="#cbd5e1", lw=1.1))
    ax.text(motor_x + motor_w / 2, 0, motor_name.split()[0], color="#e2e8f0", fontsize=7, ha="center", va="center")

    # 3. Fins
    fin_start_x = total_len_cm - fin_root_cm - 0.5
    for sign in [1.0, -1.0]:
        if fin_type == "Elliptical":
            ft = np.linspace(0.0, np.pi, 24)
            fx = fin_start_x + (fin_root_cm / 2.0) * (1.0 - np.cos(ft)) + fin_sweep_cm * np.sin(ft) * 0.5
            fy = sign * (r_cm + fin_span_cm * np.sin(ft))
            pts = np.column_stack([fx, fy])
        else:
            pts = np.array([
                [fin_start_x, sign * r_cm],
                [fin_start_x + fin_sweep_cm, sign * (r_cm + fin_span_cm)],
                [fin_start_x + fin_sweep_cm + fin_tip_cm, sign * (r_cm + fin_span_cm)],
                [fin_start_x + fin_root_cm, sign * r_cm],
            ])
        ax.add_patch(Polygon(pts, closed=True, facecolor="#2563eb", edgecolor="#bfdbfe", lw=1.5))

    ax.axhline(0.0, color="#64748b", linestyle=":", lw=1.0)

    # 4. CG (Blue) & CP (Red)
    y_arrow = r_cm + fin_span_cm * 0.45 + 1.0
    ax.scatter([cg_cm], [0.0], color="#38bdf8", s=90, zorder=6, edgecolors="white", lw=1.2)
    ax.annotate(f"CG: {cg_cm:.1f} cm", xy=(cg_cm, 0.0), xytext=(cg_cm, y_arrow),
                color="#7dd3fc", fontsize=8, fontweight="bold", ha="center",
                arrowprops=dict(arrowstyle="->", color="#7dd3fc", lw=1.0))

    ax.scatter([cp_cm], [0.0], color="#f43f5e", s=90, zorder=6, edgecolors="white", lw=1.2)
    ax.annotate(f"CP: {cp_cm:.1f} cm", xy=(cp_cm, 0.0), xytext=(cp_cm, -(y_arrow + 0.8)),
                color="#fda4af", fontsize=8, fontweight="bold", ha="center",
                arrowprops=dict(arrowstyle="->", color="#fda4af", lw=1.0))

    info_txt = (
        f"Length: {total_len_cm:.1f} cm | Dia: {body_diam_cm:.1f} cm\n"
        f"Mass: {dry_mass_g:.0f} g | Stab: {stability_cal:.2f} cal | Apogee: ~{apogee_m:.1f} m"
    )
    ax.text(0.99, 0.95, info_txt, transform=ax.transAxes, fontsize=7.5, color="#f8fafc",
            ha="right", va="top", bbox=dict(boxstyle="round,pad=0.35", facecolor="#1e293b", edgecolor="#475569"))

    y_lim = max(r_cm + fin_span_cm + 3.2, 7.5)
    ax.set_xlim(-2.0, total_len_cm + 3.0)
    ax.set_ylim(-y_lim, y_lim)
    ax.set_xlabel("Axial Position from Nose Tip (cm)", color="#cbd5e1", fontsize=8.5)
    ax.set_ylabel("Radial (cm)", color="#cbd5e1", fontsize=8.5)
    ax.set_title("2D Rocket Layout, Internal Structure & CG / CP", color="#f1f5f9", fontsize=10, pad=6)
    ax.tick_params(colors="#94a3b8", labelsize=7.5)
    for spine in ax.spines.values():
        spine.set_color("#334155")

    plt.tight_layout()
    return fig


def simulate_stability_over_time(
    nose_type, nose_len_cm, body_len_cm, body_diam_cm,
    fin_type, fin_root_cm, fin_tip_cm, fin_span_cm, fin_sweep_cm,
    cg_cm, cp_cm, dry_mass_g, motor_name, wind_ms
):
    m_info = MOTORS[motor_name]
    burn_t = m_info["burn_time"]
    thrust = m_info["thrust"]
    prop_mass_kg = m_info["prop_mass"]
    mass_kg = max(dry_mass_g / 1000.0, 0.03)

    base_stab = (cp_cm - cg_cm) / max(body_diam_cm, 0.5)

    fin_area_factor = ((fin_root_cm + fin_tip_cm) * 0.

_IncompleteInputError: incomplete input (2053518692.py, line 138)

In [5]:
!pip install -q gradio matplotlib numpy

import gradio as gr
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon, Rectangle
import numpy as np

MOTORS = {
    "A10 (1.88 Ns)": {"thrust": 3.2, "burn_time": 0.60, "prop_mass": 0.0025},
    "B6 (5.0 Ns)":  {"thrust": 6.5, "burn_time": 0.80, "prop_mass": 0.0055},
    "C6 (10.0 Ns)": {"thrust": 10.5, "burn_time": 1.00, "prop_mass": 0.0100},
}


def nose_profile_points(nose_type: str, x_start: float, length_cm: float, radius_cm: float, n_pts: int = 40):
    t = np.linspace(0.0, 1.0, n_pts)
    x = x_start + t * length_cm

    if nose_type == "Conical":
        y = radius_cm * t
    elif nose_type == "Parabolic":
        y = radius_cm * np.sqrt(t)
    elif nose_type == "Elliptical":
        y = radius_cm * np.sqrt(np.clip(1.0 - (1.0 - t) ** 2, 0.0, 1.0))
    else:  # Tangent Ogive
        rho = (radius_cm ** 2 + length_cm ** 2) / (2.0 * radius_cm)
        y = np.sqrt(np.clip(rho ** 2 - (length_cm * (1.0 - t)) ** 2, 0.0, None)) - (rho - radius_cm)
        y[0] = 0.0
        y[-1] = radius_cm

    return x, y


def draw_rocket_2d(
    nose_type, nose_len_cm, body_len_cm, body_diam_cm,
    fin_type, fin_root_cm, fin_tip_cm, fin_span_cm, fin_sweep_cm,
    cg_cm, cp_cm, dry_mass_g, motor_name, stability_cal, apogee_m
):
    fig, ax = plt.subplots(figsize=(10.5, 3.2), dpi=120)
    fig.patch.set_facecolor("#0f172a")
    ax.set_facecolor("#0f172a")

    r_cm = body_diam_cm / 2.0
    total_len_cm = nose_len_cm + body_len_cm

    # 1. Nose Cone
    nx, ny = nose_profile_points(nose_type, 0.0, nose_len_cm, r_cm)
    nose_poly = np.vstack([
        np.column_stack([nx, ny]),
        np.column_stack([nx[::-1], -ny[::-1]])
    ])
    ax.add_patch(Polygon(nose_poly, closed=True, facecolor="#1e293b", edgecolor="#38bdf8", lw=1.8))

    # 2. Body Tube
    ax.add_patch(
        Rectangle((nose_len_cm, -r_cm), body_len_cm, 2.0 * r_cm,
                  facecolor="#1e3a5f", edgecolor="#93c5fd", lw=1.6)
    )

    # Recovery bay & Motor mount
    chute_x = nose_len_cm + 0.15 * body_len_cm
    chute_w = 0.22 * body_len_cm
    ax.add_patch(Rectangle((chute_x, -0.65 * r_cm), chute_w, 1.3 * r_cm,
                           fill=False, linestyle="--", edgecolor="#facc15", lw=1.0))
    ax.text(chute_x + chute_w / 2, 0, "Recovery", color="#fde047", fontsize=7, ha="center", va="center")

    motor_w = min(0.24 * body_len_cm, 9.0)
    motor_x = total_len_cm - motor_w - 0.5
    ax.add_patch(Rectangle((motor_x, -0.55 * r_cm), motor_w, 1.1 * r_cm,
                           facecolor="#334155", edgecolor="#cbd5e1", lw=1.1))
    ax.text(motor_x + motor_w / 2, 0, motor_name.split()[0], color="#e2e8f0", fontsize=7, ha="center", va="center")

    # 3. Fins
    fin_start_x = total_len_cm - fin_root_cm - 0.5
    for sign in [1.0, -1.0]:
        if fin_type == "Elliptical":
            ft = np.linspace(0.0, np.pi, 24)
            fx = fin_start_x + (fin_root_cm / 2.0) * (1.0 - np.cos(ft)) + fin_sweep_cm * np.sin(ft) * 0.5
            fy = sign * (r_cm + fin_span_cm * np.sin(ft))
            pts = np.column_stack([fx, fy])
        else:
            pts = np.array([
                [fin_start_x, sign * r_cm],
                [fin_start_x + fin_sweep_cm, sign * (r_cm + fin_span_cm)],
                [fin_start_x + fin_sweep_cm + fin_tip_cm, sign * (r_cm + fin_span_cm)],
                [fin_start_x + fin_root_cm, sign * r_cm],
            ])
        ax.add_patch(Polygon(pts, closed=True, facecolor="#2563eb", edgecolor="#bfdbfe", lw=1.5))

    ax.axhline(0.0, color="#64748b", linestyle=":", lw=1.0)

    # 4. CG (Blue) & CP (Red)
    y_arrow = r_cm + fin_span_cm * 0.45 + 1.0
    ax.scatter([cg_cm], [0.0], color="#38bdf8", s=90, zorder=6, edgecolors="white", lw=1.2)
    ax.annotate(f"CG: {cg_cm:.1f} cm", xy=(cg_cm, 0.0), xytext=(cg_cm, y_arrow),
                color="#7dd3fc", fontsize=8, fontweight="bold", ha="center",
                arrowprops=dict(arrowstyle="->", color="#7dd3fc", lw=1.0))

    ax.scatter([cp_cm], [0.0], color="#f43f5e", s=90, zorder=6, edgecolors="white", lw=1.2)
    ax.annotate(f"CP: {cp_cm:.1f} cm", xy=(cp_cm, 0.0), xytext=(cp_cm, -(y_arrow + 0.8)),
                color="#fda4af", fontsize=8, fontweight="bold", ha="center",
                arrowprops=dict(arrowstyle="->", color="#fda4af", lw=1.0))

    info_txt = (
        f"Length: {total_len_cm:.1f} cm | Dia: {body_diam_cm:.1f} cm\n"
        f"Mass: {dry_mass_g:.0f} g | Stab: {stability_cal:.2f} cal | Apogee: ~{apogee_m:.1f} m"
    )
    ax.text(0.99, 0.95, info_txt, transform=ax.transAxes, fontsize=7.5, color="#f8fafc",
            ha="right", va="top", bbox=dict(boxstyle="round,pad=0.35", facecolor="#1e293b", edgecolor="#475569"))

    y_lim = max(r_cm + fin_span_cm + 3.2, 7.5)
    ax.set_xlim(-2.0, total_len_cm + 3.0)
    ax.set_ylim(-y_lim, y_lim)
    ax.set_xlabel("Axial Position from Nose Tip (cm)", color="#cbd5e1", fontsize=8.5)
    ax.set_ylabel("Radial (cm)", color="#cbd5e1", fontsize=8.5)
    ax.set_title("2D Rocket Layout, Internal Structure & CG / CP", color="#f1f5f9", fontsize=10, pad=6)
    ax.tick_params(colors="#94a3b8", labelsize=7.5)
    for spine in ax.spines.values():
        spine.set_color("#334155")

    plt.tight_layout()
    return fig


def simulate_stability_over_time(
    nose_type, nose_len_cm, body_len_cm, body_diam_cm,
    fin_type, fin_root_cm, fin_tip_cm, fin_span_cm, fin_sweep_cm,
    cg_cm, cp_cm, dry_mass_g, motor_name, wind_ms
):
    m_info = MOTORS[motor_name]
    burn_t = m_info["burn_time"]
    thrust = m_info["thrust"]
    prop_mass_kg = m_info["prop_mass"]
    mass_kg = max(dry_mass_g / 1000.0, 0.03)

    base_stab = (cp_cm - cg_cm) / max(body_diam_cm, 0.5)

    fin_area_factor = ((fin_root_cm + fin_tip_cm) * 0.5 * fin_span_cm) / 120.0
    aspect_penalty = max(0.0, (fin_span_cm - 9.0) * 0.025)
    wind_effect = 0.03 * wind_ms

    t_flight = burn_t + 5.5
    t = np.linspace(0.0, t_flight, 220)

    burn_progress = np.clip(t / burn_t, 0.0, 1.0)
    cg_shift_cm = 0.4 * (prop_mass_kg * 1000.0) * burn_progress * 0.25

    apogee_t = burn_t + 2.2
    peak_dip = -0.22 * np.exp(-((t - apogee_t) / 0.7) ** 2)
    dynamic_mod = (
        0.12 * fin_area_factor
        - aspect_penalty
        - wind_effect * (0.4 + 0.6 * (t / t_flight))
        + peak_dip
        - (cg_shift_cm / max(body_diam_cm, 0.5)) * 0.35
    )

    stab_t = base_stab + dynamic_mod

    impulse = thrust * burn_t
    v_est = impulse / (mass_kg + 0.5 * prop_mass_kg)
    drag_factor = 1.0 + 0.04 * (body_diam_cm - 3.5) + 0.015 * fin_span_cm
    apogee_m = max(5.0, (v_est ** 2) / (2.0 * 9.81 * drag_factor))

    fig_2d = draw_rocket_2d(
        nose_type, nose_len_cm, body_len_cm, body_diam_cm,
        fin_type, fin_root_cm, fin_tip_cm, fin_span_cm, fin_sweep_cm,
        cg_cm, cp_cm, dry_mass_g, motor_name, base_stab, apogee_m
    )

    fig_t, ax_t = plt.subplots(figsize=(10.5, 3.0), dpi=120)
    fig_t.patch.set_facecolor("#0f172a")
    ax_t.set_facecolor("#0f172a")

    ax_t.axhspan(1.0, 3.0, color="#10b981", alpha=0.14, label="Target Band (1.0–3.0 cal)")
    ax_t.axhline(1.0, color="#f59e0b", linestyle="--", lw=1.1)
    ax_t.axhline(3.0, color="#38bdf8", linestyle="--", lw=1.1)
    ax_t.axhline(0.0, color="#ef4444", linestyle=":", lw=1.1)

    ax_t.plot(t, stab_t, color="#a78bfa", lw=2.2, label="Stability (cal)")
    ax_t.axvline(burn_t, color="#fb923c", linestyle="-.", lw=1.1, label=f"Burnout ({burn_t:.2f} s)")
    ax_t.axvline(apogee_t, color="#34d399", linestyle="-.", lw=1.1, label=f"Apogee (~{apogee_t:.1f} s)")

    ax_t.set_xlabel("Flight Time, t (s)", color="#cbd5e1", fontsize=8.5)
    ax_t.set_ylabel("Stability (cal)", color="#cbd5e1", fontsize=8.5)
    ax_t.set_title("Stability Margin Over Time", color="#f1f5f9", fontsize=10, pad=6)
    ax_t.tick_params(colors="#94a3b8", labelsize=7.5)
    ax_t.legend(loc="upper right", fontsize=7, facecolor="#1e293b", edgecolor="#475569", labelcolor="#f8fafc", ncol=2)
    for spine in ax_t.spines.values():
        spine.set_color("#334155")

    plt.tight_layout()

    min_stab = float(np.min(stab_t))
    max_stab = float(np.max(stab_t))

    if min_stab < 0.8 or base_stab < 0.8:
        status = "🔴 UNSTABLE"
        comment = (
            f"**Status — Unstable:** Static stability is **{base_stab:.2f} cal** (min **{min_stab:.2f} cal**). "
            f"CP is too close to or ahead of CG. Move CG forward (nose ballast) or increase fin area/shift aft."
        )
    elif base_stab > 3.2 or max_stab > 3.5:
        status = "🟡 BORDERLINE (Over-Stable)"
        comment = (
            f"**Status — Borderline (Over-Stable):** Static stability is **{base_stab:.2f} cal** (max **{max_stab:.2f} cal**). "
            f"High stability increases wind weathercocking at **{wind_ms:.1f} m/s**. Aim for **1.5–2.5 cal**."
        )
    else:
        status = "🟢 GOOD (Nominal Stability)"
        comment = (
            f"**Status — Good:** Static stability is **{base_stab:.2f} cal** (**{min_stab:.2f} – {max_stab:.2f} cal** during flight). "
            f"**{nose_type}** nose, **{fin_type}** fins, and **{motor_name}** provide balanced stability."
        )

    return fig_2d, fig_t, f"### {status}\n\n{comment}"


with gr.Blocks() as demo:
    gr.Markdown("## 🚀 2D Rocket Design, CG/CP Visualization & Time-Dependent Stability")

    with gr.Row():
        with gr.Column(scale=1):
            nose_type = gr.Dropdown(["Ogive", "Conical", "Parabolic", "Elliptical"], value="Ogive", label="Nose Cone Shape")
            nose_len_cm = gr.Slider(5.0, 35.0, value=16.0, step=0.5, label="Nose Cone Length (cm)")
            body_len_cm = gr.Slider(15.0, 80.0, value=45.0, step=1.0, label="Body Tube Length (cm)")
            body_diam_cm = gr.Slider(2.0, 12.0, value=4.2, step=0.1, label="Body Diameter (cm)")

            fin_type = gr.Dropdown(["Trapezoidal", "Elliptical"], value="Trapezoidal", label="Fin Shape")
            fin_root_cm = gr.Slider(3.0, 20.0, value=9.0, step=0.5, label="Fin Root Chord (cm)")
            fin_tip_cm = gr.Slider(1.0, 15.0, value=4.0, step=0.5, label="Fin Tip Chord (cm)")
            fin_span_cm = gr.Slider(2.0, 18.0, value=7.5, step=0.5, label="Fin Span (cm)")
            fin_sweep_cm = gr.Slider(0.0, 15.0, value=3.0, step=0.5, label="Fin Sweep Length (cm)")

            cg_cm = gr.Slider(5.0, 90.0, value=31.0, step=0.5, label="CG from Nose (cm)")
            cp_cm = gr.Slider(5.0, 100.0, value=41.5, step=0.5, label="CP from Nose (cm)")
            dry_mass_g = gr.Slider(40.0, 600.0, value=145.0, step=5.0, label="Dry Mass (g)")
            motor_name = gr.Dropdown(list(MOTORS.keys()), value="B6 (5.0 Ns)", label="Motor Selection")
            wind_ms = gr.Slider(0.0, 15.0, value=3.5, step=0.5, label="Wind Speed (m/s)")

            run_btn = gr.Button("Run Simulation & Plot", variant="primary")

        with gr.Column(scale=2):
            plot_2d = gr.Plot(label="2D Rocket Cross-Section & CG/CP")
            plot_time = gr.Plot(label="Stability Over Time")
            analysis_out = gr.Markdown()

    inputs = [
        nose_type, nose_len_cm, body_len_cm, body_diam_cm,
        fin_type, fin_root_cm, fin_tip_cm, fin_span_cm, fin_sweep_cm,
        cg_cm, cp_cm, dry_mass_g, motor_name, wind_ms
    ]

    run_btn.click(fn=simulate_stability_over_time, inputs=inputs, outputs=[plot_2d, plot_time, analysis_out])
    demo.load(fn=simulate_stability_over_time, inputs=inputs, outputs=[plot_2d, plot_time, analysis_out])

demo.launch(share=True, theme=gr.themes.Soft())

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://796a50a3ec20538d49.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [6]:
!pip install -q gradio matplotlib numpy scipy

import gradio as gr
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon, Rectangle
import numpy as np
from scipy.integrate import solve_ivp

# Representative solid rocket motor thrust curves: (time_s, thrust_N) and propellant mass (kg)
MOTOR_PROFILES = {
    "Estes A8-3 (2.5 Ns)": {
        "t": np.array([0.0, 0.05, 0.15, 0.45, 0.75, 0.85]),
        "F": np.array([0.0, 11.8, 4.8,  4.5,  3.2,  0.0]),
        "prop_mass_kg": 0.0031,
    },
    "Estes B6-4 (5.0 Ns)": {
        "t": np.array([0.0, 0.05, 0.18, 0.55, 0.80, 0.90]),
        "F": np.array([0.0, 14.2, 6.2,  5.8,  4.0,  0.0]),
        "prop_mass_kg": 0.0062,
    },
    "Estes C6-5 (10.0 Ns)": {
        "t": np.array([0.0, 0.06, 0.25, 1.15, 1.55, 1.70]),
        "F": np.array([0.0, 14.8, 6.5,  6.0,  4.2,  0.0]),
        "prop_mass_kg": 0.0124,
    },
}


def nose_upper_profile(nose_type: str, length_m: float, radius_m: float, n_pts: int = 50):
    """Returns x (m) and upper radius y (m) from nose tip (x=0)."""
    u = np.linspace(0.0, 1.0, n_pts)
    x = u * length_m

    if nose_type == "Conical":
        y = radius_m * u
    elif nose_type == "Parabolic":
        y = radius_m * np.sqrt(u)
    elif nose_type == "Elliptical":
        y = radius_m * np.sqrt(np.clip(1.0 - (1.0 - u) ** 2, 0.0, 1.0))
    else:  # Tangent Ogive
        rho = (radius_m ** 2 + length_m ** 2) / (2.0 * radius_m)
        y = np.sqrt(np.clip(rho ** 2 - (length_m * (1.0 - u)) ** 2, 0.0, None)) - (rho - radius_m)
        y[0] = 0.0
        y[-1] = radius_m

    return x, y


def compute_barrowman_cp(
    nose_type: str, nose_len_m: float, body_diam_m: float, total_len_m: float,
    fin_type: str, n_fins: int, fin_root_m: float, fin_tip_m: float,
    fin_span_m: float, fin_sweep_m: float
):
    """
    Barrowman-style normal force derivative (CN_alpha) and Center of Pressure (x_cp, m from nose tip)
    for nose cone + fin set in subsonic regime.
    """
    d = body_diam_m
    r = d / 2.0
    s_ref = np.pi * (r ** 2)

    # Nose cone normal force slope & CP location
    cn_nose = 2.0
    nose_cp_frac = {
        "Conical": 0.667,
        "Parabolic": 0.500,
        "Elliptical": 0.333,
        "Ogive": 0.466,
    }.get(nose_type, 0.466)
    x_cp_nose = nose_cp_frac * nose_len_m

    # Fin geometry & Barrowman normal force slope
    if fin_type == "Elliptical":
        # Approximate equivalent trapezoidal area & span
        s_fin = 0.5 * np.pi * (fin_root_m / 2.0) * fin_span_m
        c_r = fin_root_m
        c_t = 0.25 * fin_root_m
    else:
        s_fin = 0.5 * (fin_root_m + fin_tip_m) * fin_span_m
        c_r = fin_root_m
        c_t = fin_tip_m

    fin_start_x = total_len_m - c_r - 0.005
    # Mid-chord length at span location
    x_mid_chord = fin_start_x + 0.5 * fin_sweep_m + 0.25 * (c_r + c_t)

    # Barrowman fin CN_alpha factor (interference factor ~ 1 + r/(span + r))
    interf = 1.0 + (r / max(fin_span_m + r, 1e-4))
    denom = 1.0 + np.sqrt(1.0 + (2.0 * fin_span_m / max(c_r + c_t, 1e-4)) ** 2)
    cn_fin = interf * (4.0 * n_fins * ((fin_span_m / d) ** 2)) / denom

    # Fin CP axial position
    x_cp_fin = x_mid_chord + (1.0 / 6.0) * (c_r + c_t - (c_r * c_t) / max(c_r + c_t, 1e-6))

    cn_total = max(cn_nose + cn_fin, 1e-4)
    x_cp = (cn_nose * x_cp_nose + cn_fin * x_cp_fin) / cn_total

    # Approximate zero-lift drag coefficient Cd0
    fin_wetted = 2.0 * n_fins * s_fin
    body_wetted = np.pi * d * (total_len_m - nose_len_m) + np.pi * d * (0.5 * nose_len_m)
    wetted_ratio = (fin_wetted + body_wetted) / max(s_ref, 1e-6)
    cd0 = 0.35 + 0.0045 * wetted_ratio

    return x_cp, cn_total, cd0


def compute_mass_and_cg(
    nose_type: str, nose_len_m: float, body_len_m: float, body_diam_m: float,
    wall_thick_mm: float, mat_density_kgm3: float, payload_mass_g: float,
    payload_pos_frac: float, fin_type: str, n_fins: int,
    fin_root_m: float, fin_tip_m: float, fin_span_m: float, fin_sweep_m: float,
    motor_name: str
):
    """Computes component masses and axial Center of Gravity x_cg (m from nose tip)."""
    r = body_diam_m / 2.0
    t_wall = max(wall_thick_mm / 1000.0, 0.0005)
    total_len = nose_len_m + body_len_m

    # 1. Nose shell mass (approx surface area * thickness * density)
    nose_area = np.pi * r * np.sqrt(r ** 2 + nose_len_m ** 2)
    m_nose = nose_area * t_wall * mat_density_kgm3
    x_nose = 0.55 * nose_len_m

    # 2. Body tube shell mass
    body_area = np.pi * body_diam_m * body_len_m
    m_body = body_area * t_wall * mat_density_kgm3
    x_body = nose_len_m + 0.5 * body_len_m

    # 3. Fins mass (3 mm fin thickness assumed)
    if fin_type == "Elliptical":
        s_fin = 0.5 * np.pi * (fin_root_m / 2.0) * fin_span_m
        c_r = fin_root_m
    else:
        s_fin = 0.5 * (fin_root_m + fin_tip_m) * fin_span_m
        c_r = fin_root_m
    m_fins = n_fins * s_fin * 0.003 * mat_density_kgm3
    x_fins = total_len - c_r - 0.005 + 0.45 * c_r + 0.25 * fin_sweep_m

    # 4. Payload / recovery mass
    m_pay = max(payload_mass_g / 1000.0, 0.0)
    x_pay = nose_len_m + payload_pos_frac * body_len_m

    # 5. Motor casing + propellant mass at aft end
    prop_m = MOTOR_PROFILES[motor_name]["prop_mass_kg"]
    m_motor_dry = 0.014 + 0.5 * prop_m
    x_motor = total_len - 0.035

    m_dry = max(m_nose + m_body + m_fins + m_pay + m_motor_dry, 0.04)
    x_cg_dry = (
        m_nose * x_nose + m_body * x_body + m_fins * x_fins +
        m_pay * x_pay + m_motor_dry * x_motor
    ) / m_dry

    return m_dry, x_cg_dry, prop_m, x_motor


def simulate_flight_physics(
    nose_type, nose_len_cm, body_len_cm, body_diam_cm,
    wall_thick_mm, mat_density, payload_mass_g, payload_pos_pct,
    fin_type, n_fins, fin_root_cm, fin_tip_cm, fin_span_cm, fin_sweep_cm,
    motor_name, wind_ms
):
    nose_len_m = nose_len_cm / 100.0
    body_len_m = body_len_cm / 100.0
    body_diam_m = body_diam_cm / 100.0
    total_len_m = nose_len_m + body_len_m

    fin_root_m = fin_root_cm / 100.0
    fin_tip_m = fin_tip_cm / 100.0
    fin_span_m = fin_span_cm / 100.0
    fin_sweep_m = fin_sweep_cm / 100.0

    m_dry, x_cg_dry, prop_m, x_motor = compute_mass_and_cg(
        nose_type, nose_len_m, body_len_m, body_diam_m,
        wall_thick_mm, mat_density, payload_mass_g, payload_pos_pct / 100.0,
        fin_type, int(n_fins), fin_root_m, fin_tip_m, fin_span_m, fin_sweep_m,
        motor_name
    )

    x_cp, cn_alpha, cd0 = compute_barrowman_cp(
        nose_type, nose_len_m, body_diam_m, total_len_m,
        fin_type, int(n_fins), fin_root_m, fin_tip_m, fin_span_m, fin_sweep_m
    )

    prof = MOTOR_PROFILES[motor_name]
    t_burn_end = prof["t"][-1]
    t_sim_end = max(t_burn_end + 4.5, 5.5)

    # State vector y = [z (altitude, m), vz (vertical velocity, m/s), theta (pitch angle from vertical, rad), omega (pitch rate, rad/s)]
    # Note: theta = 0 means vertical flight; wind induces an effective crossflow angle
    s_ref = np.pi * ((body_diam_m / 2.0) ** 2)
    i_pitch = max(m_dry * ((0.28 * total_len_m) ** 2), 1e-4)

    def thrust_at(t):
        return float(np.interp(t, prof["t"], prof["F"], left=0.0, right=0.0))

    def prop_remaining(t):
        if t <= 0.0:
            return prop_m
        if t >= t_burn_end:
            return 0.0
        frac = 1.0 - (t / t_burn_end)
        return prop_m * np.clip(frac, 0.0, 1.0)

    def current_cg(t):
        m_p = prop_remaining(t)
        m_tot = m_dry + m_p
        return (m_dry * x_cg_dry + m_p * x_motor) / max(m_tot, 1e-4)

    def rhs(t, y):
        z, vz, theta, omega = y
        m_p = prop_remaining(t)
        m_tot = max(m_dry + m_p, 1e-4)
        cg_now = current_cg(t)

        # Standard atmosphere density approximation
        rho = 1.225 * np.exp(-max(z, 0.0) / 8500.0)
        speed = np.hypot(vz, wind_ms * np.sin(theta))
        q = 0.5 * rho * (speed ** 2)

        f_t = thrust_at(t)
        # Drag force opposing motion
        drag = q * s_ref * cd0
        drag_z = drag * (vz / max(speed, 1e-3))

        # Restoring moment due to static margin (CP - CG) and crosswind angle of attack
        aoa = np.arctan2(wind_ms, max(abs(vz), 1.0)) + theta * 0.25
        static_margin_m = x_cp - cg_now
        restoring_torque = -q * s_ref * cn_alpha * static_margin_m * aoa - 0.02 * omega

        az = (f_t * np.cos(theta) - drag_z) / m_tot - 9.80665
        alpha_pitch = restoring_torque / i_pitch

        # Prevent sinking below ground during simulation
        if z <= 0.0 and az < 0.0:
            z = 0.0
            vz = max(vz, 0.0)
            az = 0.0

        return [vz, az, omega, alpha_pitch]

    t_eval = np.linspace(0.0, t_sim_end, 260)
    sol = solve_ivp(rhs, (0.0, t_sim_end), [0.0, 0.0, 0.0, 0.0], t_eval=t_eval, rtol=1e-5, atol=1e-7)

    t_arr = sol.t
    z_arr = np.maximum(sol.y[0], 0.0)
    vz_arr = sol.y[1]
    mach_arr = np.abs(vz_arr) / 340.3

    cg_t_m = np.array([current_cg(ti) for ti in t_arr])
    # Slight Mach/dynamic pressure influence on effective CP
    cp_t_m = x_cp + 0.015 * body_diam_m * np.tanh(mach_arr * 2.0) - 0.002 * np.clip(wind_ms, 0.0, 15.0)
    stab_t_cal = (cp_t_m - cg_t_m) / max(body_diam_m, 1e-3)

    static_stab_cal = (x_cp - x_cg_dry) / max(body_diam_m, 1e-3)
    apogee_m = float(np.max(z_arr))
    max_mach = float(np.max(mach_arr))

    # --- 2D Engineering Layout Plot ---
    fig_2d, ax = plt.subplots(figsize=(10.5, 3.2), dpi=120)
    fig_2d.patch.set_facecolor("#0f172a")
    ax.set_facecolor("#0f172a")

    r_cm = body_diam_cm / 2.0
    total_len_cm = nose_len_cm + body_len_cm

    nx, ny = nose_upper_profile(nose_type, nose_len_cm, r_cm)
    nose_poly = np.vstack([
        np.column_stack([nx, ny]),
        np.column_stack([nx[::-1], -ny[::-1]])
    ])
    ax.add_patch(Polygon(nose_poly, closed=True, facecolor="#1e293b", edgecolor="#38bdf8", lw=1.8))
    ax.add_patch(Rectangle((nose_len_cm, -r_cm), body_len_cm, 2.0 * r_cm,
                           facecolor="#1e3a5f", edgecolor="#93c5fd", lw=1.6))

    # Fins (top & bottom stations)
    fin_start_x = total_len_cm - fin_root_cm - 0.5
    for sign in [1.0, -1.0]:
        if fin_type == "Elliptical":
            ft = np.linspace(0.0, np.pi, 24)
            fx = fin_start_x + (fin_root_cm / 2.0) * (1.0 - np.cos(ft)) + fin_sweep_cm * np.sin(ft) * 0.5
            fy = sign * (r_cm + fin_span_cm * np.sin(ft))
            pts = np.column_stack([fx, fy])
        else:
            pts = np.array([
                [fin_start_x, sign * r_cm],
                [fin_start_x + fin_sweep_cm, sign * (r_cm + fin_span_cm)],
                [fin_start_x + fin_sweep_cm + fin_tip_cm, sign * (r_cm + fin_span_cm)],
                [fin_start_x + fin_root_cm, sign * r_cm],
            ])
        ax.add_patch(Polygon(pts, closed=True, facecolor="#2563eb", edgecolor="#bfdbfe", lw=1.5))

    ax.axhline(0.0, color="#64748b", linestyle=":", lw=1.0)

    cg_cm = x_cg_dry * 100.0
    cp_cm = x_cp * 100.0
    y_arrow = r_cm + fin_span_cm * 0.45 + 1.0

    ax.scatter([cg_cm], [0.0], color="#38bdf8", s=95, zorder=6, edgecolors="white", lw=1.2)
    ax.annotate(f"CG (dry): {cg_cm:.1f} cm", xy=(cg_cm, 0.0), xytext=(cg_cm, y_arrow),
                color="#7dd3fc", fontsize=8, fontweight="bold", ha="center",
                arrowprops=dict(arrowstyle="->", color="#7dd3fc", lw=1.0))

    ax.scatter([cp_cm], [0.0], color="#f43f5e", s=95, zorder=6, edgecolors="white", lw=1.2)
    ax.annotate(f"CP (Barrowman): {cp_cm:.1f} cm", xy=(cp_cm, 0.0), xytext=(cp_cm, -(y_arrow + 0.8)),
                color="#fda4af", fontsize=8, fontweight="bold", ha="center",
                arrowprops=dict(arrowstyle="->", color="#fda4af", lw=1.0))

    summary_box = (
        f"Dry Mass: {m_dry * 1000:.1f} g | Cd0: {cd0:.2f} | CN_a: {cn_alpha:.1f}\n"
        f"Static Margin: {static_stab_cal:.2f} cal | Apogee: {apogee_m:.1f} m | Peak Mach: {max_mach:.2f}"
    )
    ax.text(0.99, 0.95, summary_box, transform=ax.transAxes, fontsize=7.5, color="#f8fafc",
            ha="right", va="top", bbox=dict(boxstyle="round,pad=0.35", facecolor="#1e293b", edgecolor="#475569"))

    y_lim = max(r_cm + fin_span_cm + 3.2, 7.5)
    ax.set_xlim(-2.0, total_len_cm + 3.0)
    ax.set_ylim(-y_lim, y_lim)
    ax.set_xlabel("Axial Station from Nose Tip (cm)", color="#cbd5e1", fontsize=8.5)
    ax.set_ylabel("Radial Offset (cm)", color="#cbd5e1", fontsize=8.5)
    ax.set_title("2D Rocket Geometry with Barrowman CP & Mass-Derived CG", color="#f1f5f9", fontsize=10, pad=6)
    ax.tick_params(colors="#94a3b8", labelsize=7.5)
    for spine in ax.spines.values():
        spine.set_color("#334155")
    plt.tight_layout()

    # --- Time-Dependent Stability Plot ---
    fig_t, ax_t = plt.subplots(figsize=(10.5, 3.0), dpi=120)
    fig_t.patch.set_facecolor("#0f172a")
    ax_t.set_facecolor("#0f172a")

    ax_t.axhspan(1.0, 3.0, color="#10b981", alpha=0.14, label="Nominal Band (1.0–3.0 cal)")
    ax_t.axhline(1.0, color="#f59e0b", linestyle="--", lw=1.1)
    ax_t.axhline(3.0, color="#38bdf8", linestyle="--", lw=1.1)
    ax_t.axhline(0.0, color="#ef4444", linestyle=":", lw=1.1)

    ax_t.plot(t_arr, stab_t_cal, color="#a78bfa", lw=2.2, label="Static Stability Margin (cal)")
    ax_t.axvline(t_burn_end, color="#fb923c", linestyle="-.", lw=1.1, label=f"Burnout ({t_burn_end:.2f} s)")

    ax_t.set_xlabel("Flight Time, t (s)", color="#cbd5e1", fontsize=8.5)
    ax_t.set_ylabel("Stability Margin (cal)", color="#cbd5e1", fontsize=8.5)
    ax_t.set_title("Time-Dependent Stability Margin (RK45 Trajectory Integration)", color="#f1f5f9", fontsize=10, pad=6)
    ax_t.tick_params(colors="#94a3b8", labelsize=7.5)
    ax_t.legend(loc="upper right", fontsize=7, facecolor="#1e293b", edgecolor="#475569", labelcolor="#f8fafc", ncol=2)
    for spine in ax_t.spines.values():
        spine.set_color("#334155")
    plt.tight_layout()

    min_s = float(np.min(stab_t_cal))
    max_s = float(np.max(stab_t_cal))

    if min_s < 0.8 or static_stab_cal < 0.8:
        badge = "🔴 UNSTABLE"
        detail = (
            f"Static stability margin is **{static_stab_cal:.2f} cal** (flight min: **{min_s:.2f} cal**). "
            f"CP is insufficiently aft of CG; pitch/yaw divergence risk is high. "
            f"Increase fin area/span, shift fins aft, or move payload/mass forward."
        )
    elif static_stab_cal > 3.2 or max_s > 3.5:
        badge = "🟡 BORDERLINE (Over-Stable)"
        detail = (
            f"Static stability margin is **{static_stab_cal:.2f} cal** (flight max: **{max_s:.2f} cal**). "
            f"High stability increases weathercocking tendency under **{wind_ms:.1f} m/s** wind. "
            f"Target **1.5–2.5 cal** for balanced wind resistance and apogee."
        )
    else:
        badge = "🟢 GOOD (Nominal Stability)"
        detail = (
            f"Static stability margin is **{static_stab_cal:.2f} cal** (**{min_s:.2f}–{max_s:.2f} cal** over simulation). "
            f"Barrowman CP (**{cp_cm:.1f} cm**) and mass-derived CG (**{cg_cm:.1f} cm**) sit in a balanced configuration."
        )

    assumptions = (
        "**Model Scope & Assumptions:** Subsonic Barrowman CP approximation, rigid-body 2-DOF pitch/vertical integration "
        "(`scipy.integrate.solve_ivp`), exponential standard atmosphere, and prescribed motor thrust curves (no parachute deployment phase)."
    )

    return fig_2d, fig_t, f"### {badge}\n\n{detail}\n\n{assumptions}"


with gr.Blocks() as demo:
    gr.Markdown("## 🚀 Physics-Based 2D Rocket Stability & Trajectory Tool (Barrowman + RK45)")

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### Airframe & Fin Geometry")
            nose_type = gr.Dropdown(["Ogive", "Conical", "Parabolic", "Elliptical"], value="Ogive", label="Nose Cone Geometry")
            nose_len_cm = gr.Slider(5.0, 35.0, value=16.0, step=0.5, label="Nose Cone Length (cm)")
            body_len_cm = gr.Slider(15.0, 80.0, value=45.0, step=1.0, label="Body Tube Length (cm)")
            body_diam_cm = gr.Slider(2.0, 12.0, value=4.2, step=0.1, label="Body Outer Diameter (cm)")

            fin_type = gr.Dropdown(["Trapezoidal", "Elliptical"], value="Trapezoidal", label="Fin Planform")
            n_fins = gr.Dropdown([3, 4], value=3, label="Number of Fins")
            fin_root_cm = gr.Slider(3.0, 20.0, value=9.0, step=0.5, label="Fin Root Chord (cm)")
            fin_tip_cm = gr.Slider(1.0, 15.0, value=4.0, step=0.5, label="Fin Tip Chord (cm)")
            fin_span_cm = gr.Slider(2.0, 18.0, value=7.5, step=0.5, label="Fin Span (cm)")
            fin_sweep_cm = gr.Slider(0.0, 15.0, value=3.0, step=0.5, label="Fin Sweep (cm)")

            gr.Markdown("### Material, Payload & Motor")
            wall_thick_mm = gr.Slider(0.8, 5.0, value=1.8, step=0.1, label="Airframe Wall Thickness (mm)")
            mat_density = gr.Slider(120.0, 2800.0, value=680.0, step=20.0, label="Structure Density (kg/m³)")
            payload_mass_g = gr.Slider(0.0, 250.0, value=25.0, step=5.0, label="Payload / Recovery Mass (g)")
            payload_pos_pct = gr.Slider(10.0, 75.0, value=25.0, step=5.0, label="Payload Axial Position in Body (%)")
            motor_name = gr.Dropdown(list(MOTOR_PROFILES.keys()), value="Estes B6-4 (5.0 Ns)", label="Motor Profile")
            wind_ms = gr.Slider(0.0, 15.0, value=3.5, step=0.5, label="Crosswind Speed (m/s)")

            run_btn = gr.Button("Simulate & Plot", variant="primary")

        with gr.Column(scale=2):
            plot_2d = gr.Plot(label="2D Rocket & CP/CG")
            plot_time = gr.Plot(label="Stability Over Time")
            analysis_out = gr.Markdown()

    inputs = [
        nose_type, nose_len_cm, body_len_cm, body_diam_cm,
        wall_thick_mm, mat_density, payload_mass_g, payload_pos_pct,
        fin_type, n_fins, fin_root_cm, fin_tip_cm, fin_span_cm, fin_sweep_cm,
        motor_name, wind_ms
    ]

    run_btn.click(fn=simulate_flight_physics, inputs=inputs, outputs=[plot_2d, plot_time, analysis_out])
    demo.load(fn=simulate_flight_physics, inputs=inputs, outputs=[plot_2d, plot_time, analysis_out])

demo.launch(share=True, theme=gr.themes.Soft())

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://9106b1773136b4c245.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
